# 🔬 Demonstração de Clusterização: K-Means vs DBSCAN

Este notebook demonstra e compara dois algoritmos de clusterização amplamente utilizados:
- **K-Means**: Clusterização baseada em centroides, requer definição prévia do número de clusters *k*
- **DBSCAN**: Clusterização baseada em densidade, detecta automaticamente o número de clusters e lida bem com ruídos/outliers

---
### 📋 Fluxo do Notebook
1. Instalação e importação das bibliotecas
2. Geração / carregamento dos dados
3. Pré-processamento
4. **K-Means** — seleção de *k* via Curva do Cotovelo e Índice de Silhueta
5. **DBSCAN** — ajuste de parâmetros
6. Comparação dos resultados


## 1. 📦 Instalação e Importação de Bibliotecas

In [ ]:
# Instalar dependências (caso necessário no Colab)
!pip install plotly scikit-learn pandas numpy -q

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings('ignore')

# Semente para reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('✅ Bibliotecas importadas com sucesso!')

## 2. 📊 Dados

> **ℹ️ Dados Dummy Ativos**  
> O notebook está usando dados sintéticos gerados automaticamente.  
> Quando você quiser usar seus dados reais, substitua o conteúdo da célula marcada como **`[SUBSTITUIR DADOS REAIS]`** abaixo.


In [ ]:
# ==============================================================
# CONFIGURAÇÃO: Escolha o tipo de dado dummy para demonstração
# ==============================================================
# Opções:
#   'blobs'      → clusters esféricos e bem separados (favorece K-Means)
#   'moons'      → clusters em formato de meia-lua (favorece DBSCAN)
#   'mixed'      → combinação de formatos variados (desafiador para ambos)

DATASET_TYPE = 'mixed'   # << Altere aqui para testar cenários diferentes
N_SAMPLES    = 600
N_FEATURES   = 2         # 2 para visualização 2D, ou mais para dados reais

# --------------------------------------------------------------
def gerar_dados_dummy(tipo: str, n_samples: int, n_features: int) -> pd.DataFrame:
    """Gera diferentes tipos de dados sintéticos para demonstração."""

    if tipo == 'blobs':
        X, _ = make_blobs(
            n_samples=n_samples,
            n_features=n_features,
            centers=4,
            cluster_std=[1.0, 1.5, 0.8, 1.2],
            random_state=RANDOM_STATE
        )

    elif tipo == 'moons':
        X, _ = make_moons(n_samples=n_samples, noise=0.08, random_state=RANDOM_STATE)

    elif tipo == 'mixed':
        # Blobs
        X1, _ = make_blobs(n_samples=int(n_samples * 0.5), centers=[
            [-6, -6], [6, 6]
        ], cluster_std=0.8, random_state=RANDOM_STATE)
        # Moons
        X2, _ = make_moons(n_samples=int(n_samples * 0.35), noise=0.07, random_state=RANDOM_STATE)
        X2 = X2 * 2 + np.array([0, 3])
        # Outliers
        X3 = np.random.uniform(-10, 10, size=(int(n_samples * 0.05), 2))

        X = np.vstack([X1, X2, X3])
    else:
        raise ValueError(f"Tipo desconhecido: {tipo}. Use 'blobs', 'moons' ou 'mixed'.")

    cols = [f'feature_{i+1}' for i in range(X.shape[1])]
    return pd.DataFrame(X, columns=cols)

# --------------------------------------------------------------
df = gerar_dados_dummy(DATASET_TYPE, N_SAMPLES, N_FEATURES)

print(f'✅ Dataset gerado: tipo="{DATASET_TYPE}" | shape={df.shape}')
df.head()

In [ ]:
# ==============================================================
# [SUBSTITUIR DADOS REAIS]
# ==============================================================
# Quando tiver seus dados prontos, descomente e adapte o bloco abaixo.
# O restante do notebook funcionará automaticamente.
#
# --- Opção A: Upload de arquivo CSV no Google Colab ---
# from google.colab import files
# uploaded = files.upload()
# import io
# filename = list(uploaded.keys())[0]
# df = pd.read_csv(io.BytesIO(uploaded[filename]))
#
# --- Opção B: Ler do Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/seu_arquivo.csv')
#
# --- Opção C: DataFrame já existente ---
# df = seu_dataframe  # Certifique-se que contém apenas colunas numéricas
#
# Defina as colunas que serão usadas na clusterização:
# FEATURE_COLS = ['coluna_1', 'coluna_2', ...]  # Deixe None para usar todas
# ==============================================================

FEATURE_COLS = None  # None = usar todas as colunas numéricas

print('ℹ️  Bloco de dados reais disponível (comentado). Usando dados dummy.')

## 3. 🔧 Pré-processamento

In [ ]:
# Selecionar features numéricas
if FEATURE_COLS is None:
    FEATURE_COLS = df.select_dtypes(include=[np.number]).columns.tolist()

X_raw = df[FEATURE_COLS].dropna()

# Padronização (z-score)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f'Features utilizadas : {FEATURE_COLS}')
print(f'Shape após limpeza  : {X_raw.shape}')
print(f'Shape padronizado   : {X_scaled.shape}')

# Estatísticas descritivas
df[FEATURE_COLS].describe().round(3)

In [ ]:
# Visualização da distribuição original dos dados
# (Apenas para datasets 2D; para N>2 usa pairplot)

df_plot = pd.DataFrame(X_scaled, columns=FEATURE_COLS)

if len(FEATURE_COLS) == 2:
    fig = px.scatter(
        df_plot,
        x=FEATURE_COLS[0],
        y=FEATURE_COLS[1],
        title='Distribuição Original dos Dados (padronizados)',
        template='plotly_white',
        color_discrete_sequence=['#636EFA'],
        opacity=0.7
    )
    fig.update_traces(marker=dict(size=6))
    fig.update_layout(height=500)
    fig.show()

elif len(FEATURE_COLS) >= 3:
    fig = px.scatter_matrix(
        df_plot,
        dimensions=FEATURE_COLS[:6],  # limita a 6 para legibilidade
        title='Matriz de Dispersão — Dados Padronizados',
        template='plotly_white',
        opacity=0.5
    )
    fig.update_layout(height=700)
    fig.show()

## 4. 🎯 K-Means
### 4.1 Seleção do Número de Clusters (*k*)

In [ ]:
# --------------------------------------------------------------
# CONFIGURAÇÃO K-Means
# --------------------------------------------------------------
K_RANGE    = range(2, 12)   # Faixa de k a avaliar
N_INIT     = 10             # Inicializações aleatórias por k
MAX_ITER   = 300

inertias          = []
silhouette_scores = []
db_scores         = []
ch_scores         = []

print('Avaliando valores de k...', end=' ')
for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=N_INIT,
                max_iter=MAX_ITER, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_scaled)

    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))
    db_scores.append(davies_bouldin_score(X_scaled, labels))
    ch_scores.append(calinski_harabasz_score(X_scaled, labels))

print('✅')

# Sugestão automática de k ótimo (maior índice de silhueta)
best_k_silhouette = list(K_RANGE)[np.argmax(silhouette_scores)]
best_k_ch         = list(K_RANGE)[np.argmax(ch_scores)]

print(f'\n🏆 k sugerido pelo Índice de Silhueta     : {best_k_silhouette}')
print(f'🏆 k sugerido pelo Índice Calinski-Harabasz: {best_k_ch}')

In [ ]:
# Curva do Cotovelo + Índice de Silhueta — subplots lado a lado

k_list = list(K_RANGE)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Curva do Cotovelo (Inércia)', 'Índice de Silhueta')
)

# --- Cotovelo ---
fig.add_trace(
    go.Scatter(x=k_list, y=inertias, mode='lines+markers',
               name='Inércia',
               line=dict(color='#636EFA', width=2.5),
               marker=dict(size=8, symbol='circle')),
    row=1, col=1
)

# --- Silhueta ---
colors_sil = ['#EF553B' if k == best_k_silhouette else '#00CC96' for k in k_list]
fig.add_trace(
    go.Bar(x=k_list, y=silhouette_scores, name='Silhueta',
           marker_color=colors_sil,
           text=[f'{v:.3f}' for v in silhouette_scores],
           textposition='outside'),
    row=1, col=2
)
fig.add_vline(x=best_k_silhouette, line_dash='dash', line_color='#EF553B',
              annotation_text=f' k={best_k_silhouette} (melhor)',
              annotation_position='top right', row=1, col=2)

fig.update_layout(
    title_text='Seleção do k Ótimo para K-Means',
    template='plotly_white',
    height=450,
    showlegend=False
)
fig.update_xaxes(title_text='Número de Clusters (k)', dtick=1)
fig.update_yaxes(title_text='Inércia (WCSS)', row=1, col=1)
fig.update_yaxes(title_text='Silhouette Score', row=1, col=2, range=[0, 1])

fig.show()

In [ ]:
# Índices complementares: Davies-Bouldin e Calinski-Harabasz

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Davies-Bouldin (↓ melhor)', 'Calinski-Harabasz (↑ melhor)')
)

fig.add_trace(
    go.Scatter(x=k_list, y=db_scores, mode='lines+markers',
               name='Davies-Bouldin',
               line=dict(color='#AB63FA', width=2.5),
               marker=dict(size=8)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=k_list, y=ch_scores, mode='lines+markers',
               name='Calinski-Harabasz',
               line=dict(color='#FFA15A', width=2.5),
               marker=dict(size=8)),
    row=1, col=2
)

fig.update_layout(
    title_text='Índices de Avaliação de Clustering — K-Means',
    template='plotly_white',
    height=420,
    showlegend=False
)
fig.update_xaxes(title_text='Número de Clusters (k)', dtick=1)
fig.show()

### 4.2 Aplicação do K-Means com o *k* Selecionado

In [ ]:
# ==============================================================
# Defina o k final aqui (ou mantenha a sugestão automática)
# ==============================================================
K_FINAL = best_k_silhouette  # << Altere se preferir outro valor

km_final = KMeans(n_clusters=K_FINAL, init='k-means++', n_init=N_INIT,
                  max_iter=MAX_ITER, random_state=RANDOM_STATE)
km_labels = km_final.fit_predict(X_scaled)

centroids_scaled = km_final.cluster_centers_

sil_final = silhouette_score(X_scaled, km_labels)
print(f'K-Means | k={K_FINAL} | Silhouette={sil_final:.4f} | Inércia={km_final.inertia_:.2f}')

In [ ]:
# Visualização dos clusters K-Means (2D)

df_km = pd.DataFrame(X_scaled, columns=FEATURE_COLS)
df_km['Cluster'] = km_labels.astype(str)

centroids_df = pd.DataFrame(centroids_scaled, columns=FEATURE_COLS)

if len(FEATURE_COLS) >= 2:
    xcol, ycol = FEATURE_COLS[0], FEATURE_COLS[1]

    fig = px.scatter(
        df_km, x=xcol, y=ycol,
        color='Cluster',
        title=f'K-Means — k={K_FINAL} | Silhouette={sil_final:.4f}',
        template='plotly_white',
        opacity=0.75,
        color_discrete_sequence=px.colors.qualitative.Bold
    )
    fig.update_traces(marker=dict(size=7))

    # Adicionar centroides
    fig.add_trace(go.Scatter(
        x=centroids_df[xcol],
        y=centroids_df[ycol],
        mode='markers',
        marker=dict(symbol='x', size=16, color='black',
                    line=dict(width=2, color='white')),
        name='Centroides'
    ))

    fig.update_layout(height=550, legend_title_text='Cluster')
    fig.show()

In [ ]:
# Diagrama de Silhueta por Cluster

sil_samples = silhouette_samples(X_scaled, km_labels)

fig = go.Figure()
y_lower = 0
colors = px.colors.qualitative.Bold

for i in range(K_FINAL):
    sil_i = np.sort(sil_samples[km_labels == i])
    size_i = len(sil_i)
    y_upper = y_lower + size_i

    fig.add_trace(go.Bar(
        x=sil_i,
        y=list(range(y_lower, y_upper)),
        orientation='h',
        name=f'Cluster {i}',
        marker_color=colors[i % len(colors)],
        showlegend=True,
        width=1
    ))
    y_lower = y_upper + 10

fig.add_vline(x=sil_final, line_dash='dash', line_color='red',
              annotation_text=f'Média={sil_final:.3f}',
              annotation_position='top right')

fig.update_layout(
    title=f'Diagrama de Silhueta — K-Means (k={K_FINAL})',
    xaxis_title='Coeficiente de Silhueta',
    yaxis_title='Amostras por Cluster',
    template='plotly_white',
    height=500,
    yaxis=dict(showticklabels=False),
    bargap=0
)
fig.show()

## 5. 🌀 DBSCAN
### 5.1 Ajuste de Parâmetros via k-Distance Plot

In [ ]:
from sklearn.neighbors import NearestNeighbors

# ==============================================================
# CONFIGURAÇÃO DBSCAN
# ==============================================================
DBSCAN_EPS         = 0.4     # << Raio de vizinhança  (ajuste se necessário)
DBSCAN_MIN_SAMPLES = 10      # << Mínimo de pontos para formar núcleo

# k-Distance Plot para auxiliar na escolha do eps
k_neighbors = DBSCAN_MIN_SAMPLES
nbrs = NearestNeighbors(n_neighbors=k_neighbors).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
k_dist = np.sort(distances[:, -1])[::-1]

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=k_dist, mode='lines',
    line=dict(color='#19D3F3', width=2),
    name='k-distância'
))
fig.add_hline(y=DBSCAN_EPS, line_dash='dash', line_color='#EF553B',
              annotation_text=f' eps={DBSCAN_EPS}',
              annotation_position='right')

fig.update_layout(
    title=f'k-Distance Plot (k={k_neighbors}) — Auxiliar para eps do DBSCAN',
    xaxis_title='Índice da Amostra (ordenado)',
    yaxis_title=f'Distância ao {k_neighbors}º Vizinho',
    template='plotly_white',
    height=430
)
fig.show()
print('💡 Dica: O eps ideal costuma estar no ponto de maior curvatura ("cotovelo") do gráfico.')

### 5.2 Aplicação do DBSCAN

In [ ]:
db = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES, metric='euclidean')
db_labels = db.fit_predict(X_scaled)

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise_db    = np.sum(db_labels == -1)
noise_pct     = n_noise_db / len(db_labels) * 100

# Silhouette apenas se houver mais de 1 cluster e sem todos os pontos como ruído
if n_clusters_db > 1:
    mask_valid = db_labels != -1
    if mask_valid.sum() > 1:
        sil_db = silhouette_score(X_scaled[mask_valid], db_labels[mask_valid])
    else:
        sil_db = np.nan
else:
    sil_db = np.nan

print(f'DBSCAN | eps={DBSCAN_EPS} | min_samples={DBSCAN_MIN_SAMPLES}')
print(f'       | Clusters encontrados : {n_clusters_db}')
print(f'       | Ruídos (outliers)    : {n_noise_db} ({noise_pct:.1f}%)')
print(f'       | Silhouette (sem ruído): {sil_db:.4f}' if not np.isnan(sil_db) else '       | Silhouette: N/A')

In [ ]:
# Visualização dos clusters DBSCAN

df_db = pd.DataFrame(X_scaled, columns=FEATURE_COLS)
df_db['Cluster'] = db_labels
df_db['Tipo'] = df_db['Cluster'].apply(lambda x: 'Ruído (−1)' if x == -1 else f'Cluster {x}')

if len(FEATURE_COLS) >= 2:
    xcol, ycol = FEATURE_COLS[0], FEATURE_COLS[1]

    color_map = {}
    palette = px.colors.qualitative.Bold
    unique_types = sorted(df_db['Tipo'].unique(), key=lambda x: (x == 'Ruído (−1)', x))
    for i, t in enumerate(unique_types):
        color_map[t] = '#888888' if t == 'Ruído (−1)' else palette[i % len(palette)]

    fig = px.scatter(
        df_db, x=xcol, y=ycol,
        color='Tipo',
        color_discrete_map=color_map,
        title=f'DBSCAN — eps={DBSCAN_EPS}, min_samples={DBSCAN_MIN_SAMPLES} | '
              f'Clusters={n_clusters_db} | Ruídos={n_noise_db} ({noise_pct:.1f}%)',
        template='plotly_white',
        opacity=0.75,
        symbol='Tipo',
        symbol_map={'Ruído (−1)': 'x'}
    )
    fig.update_traces(marker=dict(size=7))
    fig.update_layout(height=550, legend_title_text='Rótulo')
    fig.show()

## 6. ⚖️ Comparação: K-Means vs DBSCAN

In [ ]:
# Painel comparativo lado a lado dos clusters

if len(FEATURE_COLS) >= 2:
    xcol, ycol = FEATURE_COLS[0], FEATURE_COLS[1]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f'K-Means (k={K_FINAL})',
            f'DBSCAN (eps={DBSCAN_EPS}, min_s={DBSCAN_MIN_SAMPLES})'
        )
    )

    palette = px.colors.qualitative.Bold

    # K-Means
    for c in sorted(set(km_labels)):
        mask = km_labels == c
        fig.add_trace(
            go.Scatter(
                x=X_scaled[mask, 0], y=X_scaled[mask, 1],
                mode='markers', name=f'KM-{c}',
                marker=dict(color=palette[c % len(palette)], size=6, opacity=0.7)
            ), row=1, col=1
        )
    # Centroides
    fig.add_trace(
        go.Scatter(
            x=centroids_scaled[:, 0], y=centroids_scaled[:, 1],
            mode='markers', name='Centroides',
            marker=dict(symbol='x', size=14, color='black', line=dict(width=2))
        ), row=1, col=1
    )

    # DBSCAN
    for i, c in enumerate(sorted(set(db_labels))):
        mask = db_labels == c
        is_noise = c == -1
        fig.add_trace(
            go.Scatter(
                x=X_scaled[mask, 0], y=X_scaled[mask, 1],
                mode='markers',
                name='Ruído' if is_noise else f'DB-{c}',
                marker=dict(
                    color='#AAAAAA' if is_noise else palette[i % len(palette)],
                    size=5 if is_noise else 6,
                    symbol='x' if is_noise else 'circle',
                    opacity=0.5 if is_noise else 0.75
                )
            ), row=1, col=2
        )

    fig.update_layout(
        title_text='Comparação Visual: K-Means vs DBSCAN',
        template='plotly_white',
        height=540,
        showlegend=True
    )
    fig.show()

In [ ]:
# Tabela de métricas comparativas

metricas = {
    'Métrica': [
        'Algoritmo',
        'Nº de Clusters',
        'Nº de Outliers/Ruídos',
        '% de Ruído',
        'Silhouette Score',
        'Parâmetros Principais',
        'Define k a priori?',
        'Detecta outliers?',
        'Sensível à escala?',
        'Clusters não-esféricos?'
    ],
    'K-Means': [
        'K-Means',
        K_FINAL,
        '—',
        '—',
        f'{sil_final:.4f}',
        f'k={K_FINAL}',
        '✅ Sim',
        '❌ Não',
        '✅ Sim',
        '❌ Não (esféricos)'
    ],
    'DBSCAN': [
        'DBSCAN',
        n_clusters_db,
        n_noise_db,
        f'{noise_pct:.1f}%',
        f'{sil_db:.4f}' if not np.isnan(sil_db) else 'N/A',
        f'eps={DBSCAN_EPS}, min_samples={DBSCAN_MIN_SAMPLES}',
        '❌ Não',
        '✅ Sim',
        '✅ Sim',
        '✅ Sim (formas arbitrárias)'
    ]
}

df_comp = pd.DataFrame(metricas)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Métrica</b>', '<b>K-Means</b>', '<b>DBSCAN</b>'],
        fill_color=['#2C3E50', '#636EFA', '#00CC96'],
        font=dict(color='white', size=13),
        align='left',
        height=35
    ),
    cells=dict(
        values=[df_comp['Métrica'], df_comp['K-Means'], df_comp['DBSCAN']],
        fill_color=[['#F8F9FA'] * len(df_comp),
                    ['#EEF2FF'] * len(df_comp),
                    ['#E8FBF5'] * len(df_comp)],
        align='left',
        font=dict(size=12),
        height=30
    )
)])

fig.update_layout(
    title='📊 Comparação de Métricas e Características',
    template='plotly_white',
    height=450
)
fig.show()

In [ ]:
# Distribuição de tamanho dos clusters — K-Means vs DBSCAN

km_counts = pd.Series(km_labels).value_counts().sort_index()
db_counts = pd.Series(db_labels).value_counts().sort_index()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Tamanho dos Clusters — K-Means', 'Tamanho dos Clusters — DBSCAN')
)

palette = px.colors.qualitative.Bold

fig.add_trace(
    go.Bar(
        x=[f'Cluster {i}' for i in km_counts.index],
        y=km_counts.values,
        marker_color=[palette[i % len(palette)] for i in range(len(km_counts))],
        text=km_counts.values, textposition='outside',
        name='K-Means'
    ), row=1, col=1
)

db_labels_str = ['Ruído' if i == -1 else f'Cluster {i}' for i in db_counts.index]
db_colors = ['#AAAAAA' if i == -1 else palette[j % len(palette)]
             for j, i in enumerate(db_counts.index)]
fig.add_trace(
    go.Bar(
        x=db_labels_str,
        y=db_counts.values,
        marker_color=db_colors,
        text=db_counts.values, textposition='outside',
        name='DBSCAN'
    ), row=1, col=2
)

fig.update_layout(
    title_text='Distribuição do Tamanho dos Clusters',
    template='plotly_white',
    height=430,
    showlegend=False
)
fig.update_yaxes(title_text='Número de Pontos')
fig.show()

## 7. 📝 Conclusão

| | **K-Means** | **DBSCAN** |
|---|---|---|
| **Quando usar** | Clusters esféricos, tamanho similar, sem muitos outliers | Dados com formas arbitrárias, presença de ruído/outliers |
| **Vantagem principal** | Simples, escalável, interpretável | Não exige k a priori, robusto a outliers |
| **Desvantagem principal** | Exige definição de k, sensível a outliers | Sensível aos parâmetros eps e min_samples |
| **Silhouette Score** | Alto em blobs bem separados | Alto em dados com formas não-esféricas |

---

**Próximos passos com seus dados reais:**
1. Substitua o bloco `[SUBSTITUIR DADOS REAIS]` com seu DataFrame
2. Ajuste `FEATURE_COLS` para selecionar as colunas relevantes
3. Ajuste `K_RANGE`, `DBSCAN_EPS` e `DBSCAN_MIN_SAMPLES` conforme os resultados
4. Use o k-Distance Plot para calibrar o `eps` do DBSCAN
